# LexiFlow - Apendice Analitico do Case

Este notebook acompanha o projeto como material complementar para avaliacao tecnica.

A intencao aqui nao e repetir a interface Flask, mas registrar de forma objetiva o raciocinio analitico por tras da solucao:

- contexto do problema de negocio
- validacao estrutural do dataset
- leitura inicial do corpus
- justificativa da estrategia de classificacao
- ponte entre analise, baseline e camada GenAI

## 1. Contexto do problema

A XPTO Data Solutions recebe diariamente registros textuais vindos de canais como email, chat, portal, formulários e sistemas internos. Esses textos incluem chamados de suporte, solicitacoes de clientes, relatos operacionais e feedbacks sobre servicos.

Sem categorizacao automatica, esse fluxo gera quatro dores principais:

- dificuldade de priorizacao
- alto tempo de resposta
- pouca visibilidade por tipo de problema
- dependencia excessiva de leitura manual

A proposta do LexiFlow e transformar esse fluxo em uma cadeia analitica e operacional mais clara: ingestao, leitura exploratoria, baseline supervisionado, refinamento contextual e decisao assistida.

## 2. Leitura da base de demonstracao

A celula seguinte localiza o dataset demo de forma robusta, independentemente do diretorio a partir do qual o notebook for executado. Em seguida, carrega a base em `pandas` e padroniza a coluna `texto` para evitar ruido logo na primeira leitura.

Essa etapa existe para garantir reproducibilidade do material compartilhado e para deixar explicito qual base esta sendo usada na analise.

In [1]:
from pathlib import Path

import pandas as pd

EXPECTED_COLUMNS = [
    "id_registro",
    "texto",
    "canal_origem",
    "data",
    "classe_macro",
    "classe_detalhada",
]

project_root = None
dataset_path = None

for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "data" / "demo" / "lexiflow_demo_dataset.csv"
    if candidate.exists():
        project_root = root
        dataset_path = candidate
        break

if dataset_path is None:
    raise FileNotFoundError(
        "Nao foi possivel localizar data/demo/lexiflow_demo_dataset.csv a partir do diretorio atual."
    )

df = pd.read_csv(dataset_path)
df["texto"] = df["texto"].fillna("").astype(str).str.strip()

print(f"Projeto localizado em: {project_root}")
print(f"Dataset localizado em: {dataset_path}")
print(f"Shape do dataset: {df.shape}")

df.head()

Projeto localizado em: c:\NLP\Lexi-Flow
Dataset localizado em: c:\NLP\Lexi-Flow\data\demo\lexiflow_demo_dataset.csv
Shape do dataset: (1200, 6)


,id_registro,texto,canal_origem,data,classe_macro,classe_detalhada
0,1,Erro recorrente no sistema ao tentar acessar d...,sistema,2025-03-08,Problemas Técnicos,Erro de Sistema / Aplicação
1,2,Pedido de integração para sincronização de dados.,email,2025-03-22,Solicitações Operacionais,Integração com Sistemas Externos
2,3,Aguardando posicionamento.,formulario,2025-03-02,Outros,Mensagem Genérica
3,4,O sistema apresentou um erro inesperado ao exe...,chat,2025-03-17,Problemas Técnicos,Erro de Sistema / Aplicação
4,5,Aguardando posicionamento.,chat,2025-03-23,Outros,Mensagem Genérica


## 3. Validacao estrutural do dataset

Antes de qualquer modelagem, o primeiro criterio e confirmar se a base atende ao schema esperado pelo fluxo.

A celula abaixo confronta a estrutura carregada com as colunas obrigatorias do projeto. Isso justifica tecnicamente a etapa de ingestao do app: sem schema minimo controlado, qualquer analise posterior fica fragilizada.

In [2]:
missing_columns = [column for column in EXPECTED_COLUMNS if column not in df.columns]

schema_summary = pd.DataFrame(
    {
        "coluna_esperada": EXPECTED_COLUMNS,
        "presente_no_dataset": [column in df.columns for column in EXPECTED_COLUMNS],
    }
)

print("Validacao de schema")
print(schema_summary.to_string(index=False))

if missing_columns:
    raise ValueError(f"Colunas obrigatorias ausentes: {missing_columns}")

schema_summary

Validacao de schema
 coluna_esperada  presente_no_dataset
     id_registro                 True
           texto                 True
    canal_origem                 True
            data                 True
    classe_macro                 True
classe_detalhada                 True


,coluna_esperada,presente_no_dataset
0,id_registro,True
1,texto,True
2,canal_origem,True
3,data,True
4,classe_macro,True
5,classe_detalhada,True


## 4. Visao executiva do volume e da taxonomia

Com a estrutura validada, o proximo passo e medir rapidamente o tamanho do problema: quantidade de registros, numero de macroclasses, numero de classes detalhadas e variedade de canais.

Essa leitura e importante porque mostra se o caso tem diversidade suficiente para justificar classificacao supervisionada e se a taxonomia ja apresenta sinais de complexidade operacional.

In [3]:
overview = pd.DataFrame(
    [
        {"indicador": "total_registros", "valor": int(df.shape[0])},
        {"indicador": "total_colunas", "valor": int(df.shape[1])},
        {"indicador": "macroclasses", "valor": int(df["classe_macro"].nunique())},
        {"indicador": "classes_detalhadas", "valor": int(df["classe_detalhada"].nunique())},
        {"indicador": "canais_origem", "valor": int(df["canal_origem"].nunique())},
    ]
)

overview

,indicador,valor
0,total_registros,1200
1,total_colunas,6
2,macroclasses,5
3,classes_detalhadas,26
4,canais_origem,4


## 5. Distribuicoes operacionais

A celula seguinte resume tres distribuicoes centrais para a leitura do case:

- distribuicao por `classe_macro`
- distribuicao por `canal_origem`
- classes detalhadas mais frequentes

O objetivo aqui e verificar equilibrio, recorrencia e concentracao de volume. Esses sinais ajudam a justificar tanto a EDA do app quanto a decisao de modelar o problema em niveis hierarquicos.

In [4]:
macro_distribution = (
    df["classe_macro"]
    .value_counts()
    .rename_axis("classe_macro")
    .reset_index(name="registros")
)

channel_distribution = (
    df["canal_origem"]
    .value_counts()
    .rename_axis("canal_origem")
    .reset_index(name="registros")
)

detailed_distribution = (
    df["classe_detalhada"]
    .value_counts()
    .rename_axis("classe_detalhada")
    .reset_index(name="registros")
)

print("Distribuicao por macroclasse")
print(macro_distribution.to_string(index=False))
print()
print("Distribuicao por canal de origem")
print(channel_distribution.to_string(index=False))
print()
print("Top classes detalhadas")
print(detailed_distribution.head(10).to_string(index=False))

macro_distribution

Distribuicao por macroclasse
             classe_macro  registros
                   Outros        250
Solicitações Operacionais        244
    Financeiro e Cobrança        238
   Feedback e Experiência        237
       Problemas Técnicos        231

Distribuicao por canal de origem
canal_origem  registros
       email        310
        chat        305
  formulario        302
     sistema        283

Top classes detalhadas
               classe_detalhada  registros
          Agradecimento Simples         84
            Registro Automático         83
              Mensagem Genérica         83
         Crítica de Usabilidade         76
Sugestão de Nova Funcionalidade         66
              Feedback Positivo         53
  Cobrança Indevida / Duplicada         46
    Erro de Sistema / Aplicação         43
           Sugestão de Melhoria         42
  Erro de Autenticação e Acesso         42


,classe_macro,registros
0,Outros,250
1,Solicitações Operacionais,244
2,Financeiro e Cobrança,238
3,Feedback e Experiência,237
4,Problemas Técnicos,231


## 6. Perfil textual do corpus

Nem todo problema de classificacao textual depende apenas de rótulos. O formato do texto tambem importa.

A celula abaixo calcula metricas simples de comprimento e exibe exemplos por macroclasse. Isso ajuda a responder duas perguntas importantes:

- o corpus parece curto, objetivo e operacional ou longo e muito narrativo?
- existem sinais de ambiguidade ou sobreposicao entre classes ainda na leitura manual?

Essas observacoes justificam por que um baseline estatistico pode ser suficiente como primeiro passo e em que ponto a GenAI passa a agregar contexto.

In [5]:
df["qtd_caracteres"] = df["texto"].str.len()
df["qtd_palavras"] = df["texto"].str.split().str.len()

text_metrics = pd.DataFrame(
    [
        {"indicador": "media_caracteres", "valor": round(float(df["qtd_caracteres"].mean()), 2)},
        {"indicador": "media_palavras", "valor": round(float(df["qtd_palavras"].mean()), 2)},
        {"indicador": "menor_texto", "valor": int(df["qtd_caracteres"].min())},
        {"indicador": "maior_texto", "valor": int(df["qtd_caracteres"].max())},
    ]
)

sample_by_macro = (
    df[["classe_macro", "classe_detalhada", "canal_origem", "texto"]]
    .groupby("classe_macro", group_keys=False)
    .head(2)
    .reset_index(drop=True)
)

print("Metricas textuais")
print(text_metrics.to_string(index=False))
print()
print("Exemplos por macroclasse")
print(sample_by_macro.to_string(index=False))

text_metrics

Metricas textuais
       indicador  valor
media_caracteres  40.37
  media_palavras   5.64
     menor_texto  20.00
     maior_texto  95.00

Exemplos por macroclasse
             classe_macro                       classe_detalhada canal_origem                                                                            texto
       Problemas Técnicos            Erro de Sistema / Aplicação      sistema       Erro recorrente no sistema ao tentar acessar determinadas funcionalidades.
Solicitações Operacionais       Integração com Sistemas Externos        email                                Pedido de integração para sincronização de dados.
                   Outros                      Mensagem Genérica   formulario                                                       Aguardando posicionamento.
       Problemas Técnicos            Erro de Sistema / Aplicação         chat O sistema apresentou um erro inesperado ao executar a funcionalidade solicitada.
                   Outros                

,indicador,valor
0,media_caracteres,40.37
1,media_palavras,5.64
2,menor_texto,20.00
3,maior_texto,95.00


## 7. Hipoteses de modelagem

A leitura exploratoria acima sustenta algumas hipoteses que orientam a estrategia do projeto:

- a macroclasse tende a ser mais estavel que a classe detalhada
- a classe detalhada deve melhorar quando o espaco de classes e restringido pela macro
- textos curtos e ambiguos tendem a concentrar parte relevante dos erros
- canais distintos podem introduzir contextos diferentes mesmo com taxonomia comum
- a camada GenAI faz mais sentido como refinamento contextual do que como classificador principal

Essas hipoteses sao coerentes com a arquitetura apresentada no app.

## 8. Justificativa do baseline

A escolha por `TF-IDF + Logistic Regression` foi intencional.

Para este case, essa combinacao entrega um baseline forte porque:

- treina rapido
- e reproduzivel
- funciona bem com features esparsas
- permite avaliacao clara por metricas e matriz de confusao
- facilita a defesa tecnica da solucao antes de introduzir camadas mais sofisticadas

O foco aqui e demonstrar criterio na estrategia de classificacao, e nao apenas complexidade tecnica.

## 9. Por que modelar em dois niveis

O problema foi estruturado de forma hierarquica por aderencia ao negocio.

Na pratica, faz mais sentido responder duas perguntas em sequencia:

1. qual e a natureza do caso?
2. qual e o motivo detalhado dentro dessa natureza?

Essa decisao reduz o espaco de erro no detalhamento, aproxima o modelo da taxonomia operacional e prepara melhor o encaixe da camada generativa.

## 10. Como este apendice conversa com o app

As leituras feitas aqui aparecem de forma operacional nas rotas principais do projeto:

- `/eda` transforma estas distribuicoes e metricas em leitura visual do corpus
- `/baseline` mostra as metricas supervisionadas, matrices de confusao e leitura critica do erro
- `/predict` mostra o fluxo ponta a ponta, incluindo macro prevista, classe detalhada, provider usado e recomendacao operacional
- `/genai-demo` deixa explicito que a GenAI entra como refinamento complementar

Com isso, o notebook serve como respaldo analitico e o app como demonstracao de produto.

## 11. Perguntas provaveis da avaliacao

Algumas perguntas que este material ajuda a responder:

- por que comecar com um baseline supervisionado?
- por que nao usar um LLM como classificador principal?
- por que a modelagem hierarquica faz sentido para o negocio?
- onde os erros do baseline tendem a aparecer?
- o que faltaria para evoluir a solucao para um cenario de producao?

Essas perguntas sao relevantes porque avaliam nao apenas implementacao, mas tambem criterio analitico, entendimento de negocio e capacidade de justificar decisoes tecnicas.